# 02 — Classical ML Baselines
## Wind Turbine Gearbox Anomaly Detection

Bu notebook, makine öğrenmesinin klasik yöntemlerini kullanarak anomali tespiti için güçlü bir baseline oluşturur.

**Kullanılan Modeller:**
- Random Forest
- XGBoost
- LightGBM
- Logistic Regression

**Temel Prensipler:**
- Temporal split (zaman bazlı): veri sırasına göre train/test, shuffle yok
- SMOTE ile class imbalance yönetimi
- Threshold optimizasyonu
- Feature importance analizi

## 1. Kurulum ve Kütüphaneler

In [ ]:
# !pip install xgboost lightgbm imbalanced-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, f1_score, precision_score,
    recall_score, average_precision_score, roc_curve
)
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import lightgbm as lgb

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')
print('Libraries loaded successfully!')

# Ensure results directory exists
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to: {os.path.abspath(RESULTS_DIR)}")


## 2. Veri Yükleme ve Feature Engineering Pipeline

01_EDA_Feature_Engineering notebook'undan üretilen engineered features'ları kullanıyoruz.

In [ ]:
# Engineered features yükle
PROCESSED_PATH = '../01_EDA_Feature_Engineering/results/features_engineered.csv'
DATA_PATH = '/kaggle/input/wind-turbine-gearbox-anomaly-detection-5year-scada/'

if os.path.exists(PROCESSED_PATH):
    df = pd.read_csv(PROCESSED_PATH, index_col=0, parse_dates=True)
    print(f'Loaded engineered features: {df.shape}')
else:
    # Ham veri yükle ve hızlı feature engineering uygula
    import glob
    csv_files = glob.glob(os.path.join(DATA_PATH, '*.csv'))
    dfs = [pd.read_csv(f) for f in sorted(csv_files)]
    if not csv_files:
        raise FileNotFoundError(
            f'No CSV files found in {DATA_PATH}. '
            'Run notebook 01 first or ensure the dataset is mounted.'
        )
    df = pd.concat(dfs, ignore_index=True)
    
    # Zaman sütunu
    time_col = [c for c in df.columns if 'time' in c.lower() or 'date' in c.lower()]
    if time_col:
        df[time_col[0]] = pd.to_datetime(df[time_col[0]])
        df = df.sort_values(time_col[0]).set_index(time_col[0])
    
    print(f'Loaded raw data: {df.shape}')

# Anomali sütununu belirle
anomaly_col = [c for c in df.columns if 'anomal' in c.lower() or 'label' in c.lower() or 'fault' in c.lower() or 'alarm' in c.lower() or 'fail' in c.lower() or 'error' in c.lower() or 'status' in c.lower()]
ANOMALY_COL = anomaly_col[0] if anomaly_col else df.columns[-1]
# Ensure the anomaly column is binary (0/1);
# if values are continuous/multi-class, binarize: any non-zero → 1
_unique = df[ANOMALY_COL].dropna().unique()
if not set(_unique).issubset({0, 1, 0.0, 1.0, True, False}):
    print(f'Warning: {ANOMALY_COL!r} has non-binary values {sorted(_unique)[:5]}...'
          ' — binarizing (0=normal, >0=anomaly).')
    df[ANOMALY_COL] = (df[ANOMALY_COL] != 0).astype(int)
print(f'Anomaly column: {ANOMALY_COL}')
print(f'Class distribution:\n{df[ANOMALY_COL].value_counts()}')

## 3. Zaman Bazlı Train/Test Split (Temporal Split)

**Kritik nokta:** Zaman serisi verilerinde shuffle edilmiş random split YAPILMAZ. Bu, gelecekten geçmişi tahmin etmek anlamına gelir (data leakage). Bunun yerine kronolojik sıra korunur:
- İlk %80: Training
- Son %20: Test

In [ ]:
# Feature/target ayırma
X = df.select_dtypes(include=[np.number]).drop(columns=[ANOMALY_COL], errors='ignore')
y = df[ANOMALY_COL].astype(int)

# NaN/inf temizle
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

# Temporal split (shuffle=False)
SPLIT_RATIO = 0.8
split_idx = int(len(X) * SPLIT_RATIO)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f'Total samples:   {len(X):,}')
print(f'Training:        {len(X_train):,} ({SPLIT_RATIO*100:.0f}%)')
print(f'Test:            {len(X_test):,} ({(1-SPLIT_RATIO)*100:.0f}%)')
print(f'\nTrain anomaly rate: {y_train.mean()*100:.2f}%')
print(f'Test anomaly rate:  {y_test.mean()*100:.2f}%')

# Validation için ikincil split (train'i de böl)
val_idx = int(len(X_train) * 0.875)  # Train'in %87.5'i = toplam %70
X_tr, X_val = X_train.iloc[:val_idx], X_train.iloc[val_idx:]
y_tr, y_val = y_train.iloc[:val_idx], y_train.iloc[val_idx:]
print(f'\nFinal splits — Train: {len(X_tr):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

# Validate: both classes must be present in training split
if len(X_tr) == 0:
    raise ValueError(
        f'Training set is empty after split (X has {len(X)} rows). '
        'Check that the dataset loaded correctly.'
    )
_tr_classes = set(y_tr.unique())
if not {0, 1}.issubset(_tr_classes):
    raise ValueError(
        f'Training set is missing classes. Found: {_tr_classes}. '
        f'Anomaly column "{ANOMALY_COL}" may not be binary — '
        'check that notebook 01 was run first or the dataset path is correct.'
    )
print(f'Class check passed — y_tr: 0={( y_tr==0).sum():,} / 1={(y_tr==1).sum():,}')


## 4. SMOTE ile Class Imbalance Çözümü

SMOTE (Synthetic Minority Oversampling Technique), azınlık sınıfı için sentetik örnekler oluşturur. **Sadece training setine uygulanır** — test seti dokunulmadan bırakılır.

In [ ]:
print('Applying SMOTE on training data...')
print(f'Before SMOTE — 0: {(y_tr==0).sum():,}, 1: {(y_tr==1).sum():,}')

# Determine k_neighbors safely (must be < minority class count)
_minority_count = min((y_tr == 0).sum(), (y_tr == 1).sum())
if _minority_count < 2:
    raise ValueError(
        f'SMOTE requires at least 2 samples per class. '
        f'Minority class has {_minority_count} sample(s). '
        'Ensure the data loaded correctly and the anomaly column is binary.'
    )
_k = min(5, _minority_count - 1)
smote = SMOTE(random_state=42, k_neighbors=_k)
X_tr_resampled, y_tr_resampled = smote.fit_resample(X_tr, y_tr)

print(f'After SMOTE  — 0: {(y_tr_resampled==0).sum():,}, 1: {(y_tr_resampled==1).sum():,}')
print(f'Training size: {len(X_tr):,} → {len(X_tr_resampled):,}')

# Görselleştir
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, data, title in zip(axes,
    [y_tr, y_tr_resampled],
    ['Before SMOTE', 'After SMOTE']):
    counts = pd.Series(data).value_counts().sort_index()
    ax.bar(['Normal (0)', 'Anomaly (1)'], counts.values,
           color=['#2ecc71', '#e74c3c'], alpha=0.85)
    ax.set_title(title, fontsize=13)
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + counts.max()*0.02, f'{v:,}', ha='center')
    ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('results/smote_balancing.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Model Eğitimi

Dört farklı model eğiteceğiz: Random Forest, XGBoost, LightGBM ve Logistic Regression.

In [ ]:
def train_and_evaluate(model, model_name, X_train, y_train, X_test, y_test, threshold=0.5):
    """Modeli eğit, değerlendir ve sonuçları döndür."""
    from sklearn.preprocessing import StandardScaler
    
    # Eğit
    model.fit(X_train, y_train)
    
    # Olasılık tahminleri
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)
    
    # Metrikler
    metrics = {
        'Model': model_name,
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_proba),
        'PR-AUC': average_precision_score(y_test, y_proba)
    }
    
    return model, y_proba, y_pred, metrics

# Scaler sadece Logistic Regression için
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_resampled)
X_test_scaled = scaler.transform(X_test)

# Modeller
models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=15,
        class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=200, max_depth=7, learning_rate=0.1,
        scale_pos_weight=(y_tr_resampled==0).sum()/(y_tr_resampled==1).sum(),
        random_state=42, eval_metric='logloss', use_label_encoder=False
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=200, max_depth=7, learning_rate=0.05,
        class_weight='balanced', random_state=42, verbose=-1
    ),
    'Logistic Regression': LogisticRegression(
        class_weight='balanced', max_iter=1000, C=1.0, random_state=42
    )
}

results = []
trained_models = {}
probas = {}

for name, model in models.items():
    print(f'Training {name}...')
    # LR için scaled data kullan
    X_tr_use = X_tr_scaled if 'Logistic' in name else X_tr_resampled
    X_te_use = X_test_scaled if 'Logistic' in name else X_test
    
    trained_model, y_proba, y_pred, metrics = train_and_evaluate(
        model, name, X_tr_use, y_tr_resampled, X_te_use, y_test
    )
    results.append(metrics)
    trained_models[name] = trained_model
    probas[name] = y_proba
    print(f'  F1={metrics["F1"]:.4f} | ROC-AUC={metrics["ROC-AUC"]:.4f}')

results_df = pd.DataFrame(results).set_index('Model')
print('\n=== MODEL COMPARISON ===')
print(results_df.round(4))
# Metrikleri CSV olarak kaydet
results_df.to_csv(f"{RESULTS_DIR}/model_metrics.csv")
print("Saved: model_metrics.csv")


## 6. Confusion Matrix Görselleştirmesi

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (name, proba) in enumerate(probas.items()):
    y_pred = (proba >= 0.5).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Normal', 'Anomaly'],
                yticklabels=['Normal', 'Anomaly'])
    axes[i].set_title(f'{name} — Confusion Matrix', fontsize=12)
    axes[i].set_ylabel('Actual')
    axes[i].set_xlabel('Predicted')

plt.suptitle('Confusion Matrices (threshold=0.5)', fontsize=14)
plt.tight_layout()
plt.savefig('results/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. ROC ve PR Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

for (name, proba), color in zip(probas.items(), colors):
    # ROC curve
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[0].plot(fpr, tpr, color=color, label=f'{name} (AUC={auc:.3f})', linewidth=2)
    
    # PR curve
    precision, recall, _ = precision_recall_curve(y_test, proba)
    pr_auc = average_precision_score(y_test, proba)
    axes[1].plot(recall, precision, color=color, label=f'{name} (AP={pr_auc:.3f})', linewidth=2)

axes[0].plot([0,1], [0,1], 'k--', linewidth=1, label='Random Classifier')
axes[0].set_title('ROC Curves', fontsize=13)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

baseline_precision = y_test.mean()
axes[1].axhline(y=baseline_precision, color='k', linestyle='--',
                label=f'Baseline ({baseline_precision:.3f})')
axes[1].set_title('Precision-Recall Curves', fontsize=13)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Threshold Optimizasyonu

Varsayılan threshold=0.5 her zaman optimal değildir. F1 skorunu maksimize eden threshold'u buluyoruz.

In [ ]:
def find_best_threshold(y_true, y_proba, metric='f1'):
    """F1 veya diğer metriği maksimize eden threshold'u bul."""
    thresholds = np.arange(0.05, 0.96, 0.01)
    best_score, best_threshold = 0, 0.5
    scores = []
    
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        scores.append(score)
        if score > best_score:
            best_score = score
            best_threshold = t
    
    return best_threshold, best_score, thresholds, scores

# Threshold optimizasyonu VALIDATION seti üzerinde yapılır (test sızıntısını önlemek için)
# Önce validation seti üzerinde her modelin olasılıklarını hesapla
probas_val = {}
for name, model in models.items():
    X_val_use = scaler.transform(X_val) if 'Logistic' in name else X_val
    probas_val[name] = model.predict_proba(X_val_use)[:, 1]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
best_thresholds = {}

for i, (name, proba_val) in enumerate(probas_val.items()):
    best_t, best_f1, thresholds, scores = find_best_threshold(y_val, proba_val)
    best_thresholds[name] = best_t
    
    axes[i].plot(thresholds, scores, linewidth=2, color='steelblue')
    axes[i].axvline(x=best_t, color='red', linestyle='--',
                    label=f'Best: {best_t:.2f} (F1={best_f1:.3f})')
    axes[i].axvline(x=0.5, color='gray', linestyle=':', label='Default: 0.5')
    axes[i].set_title(f'{name} — Threshold vs F1 (Validation Set)', fontsize=12)
    axes[i].set_xlabel('Threshold')
    axes[i].set_ylabel('F1 Score')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Threshold Optimization on Validation Set (F1 Maximization)', fontsize=14)
plt.tight_layout()
plt.savefig('results/threshold_optimization.png', dpi=150, bbox_inches='tight')
plt.show()

print('Best thresholds (selected on validation set):', best_thresholds)

In [ ]:
# Optimize edilmiş threshold ile yeniden değerlendir
results_opt = []
for name, proba in probas.items():
    t = best_thresholds[name]
    y_pred = (proba >= t).astype(int)
    metrics = {
        'Model': name,
        'Threshold': t,
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1 (opt)': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, proba)
    }
    results_opt.append(metrics)

results_opt_df = pd.DataFrame(results_opt).set_index('Model')
print('=== OPTIMIZED THRESHOLD RESULTS ===')
print(results_opt_df.round(4))
# Optimize threshold sonuçlarını kaydet
results_opt_df.to_csv(f"{RESULTS_DIR}/model_metrics_optimized_threshold.csv")
print("Saved: model_metrics_optimized_threshold.csv")


## 9. Feature Importance Grafikleri (Top 20)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 8))
tree_models = ['Random Forest', 'XGBoost', 'LightGBM']
colors = ['#3498db', '#e74c3c', '#2ecc71']

for ax, name, color in zip(axes, tree_models, colors):
    model = trained_models[name]
    importances = pd.Series(model.feature_importances_, index=X_train.columns)
    top20 = importances.nlargest(20)
    
    top20.plot(kind='barh', ax=ax, color=color, alpha=0.8)
    ax.set_title(f'{name}\nTop 20 Feature Importances', fontsize=12)
    ax.set_xlabel('Importance Score')
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('results/feature_importance_models.png', dpi=150, bbox_inches='tight')
plt.show()
# Feature önemlerini CSV olarak kaydet
for name in tree_models:
    model = trained_models[name]
    imp = pd.Series(model.feature_importances_, index=X_train.columns)
    fname = name.lower().replace(" ", "_")
    imp.sort_values(ascending=False).to_frame("importance").to_csv(
        f"{RESULTS_DIR}/feature_importance_{fname}.csv"
    )
print("Saved: feature_importance_*.csv")


## 10. Karşılaştırmalı Sonuç Tablosu

In [ ]:
# Birleşik sonuç tablosu
comparison_df = pd.DataFrame(results).set_index('Model')

# Isı haritası ile görselleştir
fig, ax = plt.subplots(figsize=(10, 4))
metric_cols = ['Precision', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC']
sns.heatmap(comparison_df[metric_cols], annot=True, fmt='.4f',
            cmap='YlOrRd', ax=ax, vmin=0, vmax=1)
ax.set_title('Model Comparison — All Metrics', fontsize=14)
plt.tight_layout()
plt.savefig('results/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Styled DataFrame
print('=== FINAL MODEL COMPARISON TABLE ===')
comparison_df[metric_cols].style.background_gradient(cmap='YlOrRd').format('{:.4f}')

In [ ]:
# En iyi modeli belirle
best_model_name = comparison_df['F1'].idxmax()
print(f'Best model by F1: {best_model_name}')
print(f'  F1:      {comparison_df.loc[best_model_name, "F1"]:.4f}')
print(f'  ROC-AUC: {comparison_df.loc[best_model_name, "ROC-AUC"]:.4f}')
print(f'  PR-AUC:  {comparison_df.loc[best_model_name, "PR-AUC"]:.4f}')

# Modeli kaydet
import joblib
os.makedirs('../models', exist_ok=True)
joblib.dump(trained_models[best_model_name], f'../models/best_classical_model.pkl')
print(f'\nBest model saved to ../models/best_classical_model.pkl')
print('\n✅ Classical ML Baselines Complete!')
# Test seti tahminlerini kaydet
preds_df = pd.DataFrame(probas, index=X_test.index if hasattr(X_test, "index") else range(len(y_test)))
preds_df["y_true"] = y_test.values
preds_df.to_csv(f"{RESULTS_DIR}/test_predictions.csv")
print("Saved: test_predictions.csv")


## Özet

Bu notebook'ta:
- Temporal split ile data leakage önlendi
- SMOTE ile class imbalance çözüldü
- 4 klasik ML modeli eğitildi ve karşılaştırıldı
- Threshold optimizasyonu ile F1 skoru iyileştirildi
- Feature importance analizi yapıldı

**Sonraki Adım:** `03_Anomaly_Detection_Unsupervised` — etiket kullanmadan anomali tespiti